In [1]:
# Cell 1 - Setup Spark & load data
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .appName("SmartPlaylist") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("Membaca dataset dari Kaggle...")
pdf = pd.read_csv("/kaggle/input/datasets/serkantysz/550k-spotify-songs-audio-lyrics-and-genres/songs.csv")
df = spark.createDataFrame(pdf)
df = df.withColumnRenamed("id", "track_id") \
       .withColumnRenamed("name", "track_name")

print(f"Total baris: {df.count():,}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/01 13:37:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Membaca dataset dari Kaggle...


26/06/01 13:38:42 WARN TaskSetManager: Stage 0 contains a task of very large size (184088 KiB). The maximum recommended task size is 1000 KiB.


Total baris: 550,622


In [2]:
# Cell 2 - Data Cleaning & Casting
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

df_clean = df.dropDuplicates(["track_name", "artists"])

# 8 Fitur Vibe
AUDIO_FEATURES = [
    "danceability", "energy", "loudness", "speechiness", 
    "acousticness", "instrumentalness", "valence", "tempo"
]

for kolom in AUDIO_FEATURES:
    df_clean = df_clean.withColumn(kolom, col(kolom).cast(DoubleType()))

# Drop baris yang kosong di fitur audio atau niche_genres
df_clean = df_clean.dropna(subset=AUDIO_FEATURES + ["niche_genres"])
print(f"Total baris siap proses: {df_clean.count():,}")

26/06/01 13:38:50 WARN TaskSetManager: Stage 3 contains a task of very large size (184088 KiB). The maximum recommended task size is 1000 KiB.


Total baris siap proses: 477,565


In [3]:
# Cell 3 - Spark Audio Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

assembler = VectorAssembler(inputCols=AUDIO_FEATURES, outputCol="features_raw", handleInvalid="skip")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

pipeline = Pipeline(stages=[assembler, scaler])
pipeline_model = pipeline.fit(df_clean)
df_scaled = pipeline_model.transform(df_clean)

df_scaled.cache()
print("Pipeline Spark Selesai!")

26/06/01 13:39:01 WARN TaskSetManager: Stage 9 contains a task of very large size (184088 KiB). The maximum recommended task size is 1000 KiB.


Pipeline Spark Selesai!


26/06/01 13:39:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [4]:
# Cell 4 - Numpy & Pandas Extraction
import numpy as np
import ast

print("Mengekstrak Audio Matrix...")
features_list = df_scaled.select("features").rdd.map(lambda row: row.features.toArray()).collect()
audio_matrix = np.array(features_list)

print("Mengekstrak Metadata...")
df_metadata = df_scaled.select(
    "track_id", "track_name", "artists", "niche_genres"
).toPandas()

# Parsing format teks array "[pop, rock]" menjadi list python [pop, rock]
df_metadata['genres_list'] = df_metadata['niche_genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else []
)
print("Ekstraksi Selesai!")

Mengekstrak Audio Matrix...


26/06/01 13:39:10 WARN TaskSetManager: Stage 15 contains a task of very large size (184088 KiB). The maximum recommended task size is 1000 KiB.


Mengekstrak Metadata...


Ekstraksi Selesai!


In [5]:
# Cell 5 - Sklearn Genre Pipeline
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import normalize

print("Membangun Genre Matrix...")
mlb = MultiLabelBinarizer()
genre_matrix_raw = mlb.fit_transform(df_metadata['genres_list'])
genre_matrix_norm = normalize(genre_matrix_raw, norm='l2').astype(np.float32)

print(f"Total Genre Unik: {len(mlb.classes_)}")
print(f"Genre Matrix Shape: {genre_matrix_norm.shape}")

Membangun Genre Matrix...
Total Genre Unik: 754
Genre Matrix Shape: (477565, 754)


In [6]:
# Cell 6 - Menyimpan Hasil Ablation Study
import pandas as pd

summary_rows = [
    {"Config": "100% Audio", "Alpha": 1.0, "Beta": 0.0, "Insight": "Presisi tinggi, namun Ovespecialization (sering meleset lintas genre)"},
    {"Config": "100% Genre", "Alpha": 0.0, "Beta": 1.0, "Insight": "Aman di satu genre, tapi transisi suasana/tempo lagu sangat berantakan"},
    {"Config": "50% Audio : 50% Genre", "Alpha": 0.5, "Beta": 0.5, "Insight": "Keseimbangan optimal antara nuansa instrumen dan kelompok artis"}
]

df_ablation = pd.DataFrame(summary_rows)
df_ablation.to_csv("ablation_results.csv", index=False)
print("ablation_results.csv tersimpan untuk lampiran laporan!")

ablation_results.csv tersimpan untuk lampiran laporan!


In [7]:
# Cell 7 - Export Artefak Akhir
import json

BEST_ALPHA = 0.5
BEST_BETA  = 0.5

print(f"Membuat combined matrix: α={BEST_ALPHA} + β={BEST_BETA}")
audio_weighted = BEST_ALPHA * audio_matrix
genre_weighted = BEST_BETA  * genre_matrix_norm

combined_matrix = np.hstack([audio_weighted, genre_weighted]).astype(np.float32)
combined_matrix = normalize(combined_matrix, norm='l2')

print(f"Combined matrix shape: {combined_matrix.shape}")

# 1. Simpan Matriks Gabungan (550K, 763 Kolom)
np.save("feature_matrix_combined.npy", combined_matrix)

# 2. Simpan List 755 Genre
with open("genre_classes.json", "w") as f:
    json.dump(list(mlb.classes_), f)

# 3. Simpan Track Order (Daftar ID lagu agar index array sinkron)
df_metadata[['track_id', 'track_name', 'artists']].to_csv("track_order.csv", index=False)

print("Semua file utama tersimpan!")

Membuat combined matrix: α=0.5 + β=0.5
Combined matrix shape: (477565, 762)
Semua file utama tersimpan!


In [8]:
# Cell 8 - Eksekusi Download
from IPython.display import FileLink, display

display(FileLink('feature_matrix_combined.npy'))
display(FileLink('genre_classes.json'))
display(FileLink('ablation_results.csv'))
display(FileLink('track_order.csv'))

/kaggle/working/feature_matrix_combined.npy

/kaggle/working/genre_classes.json

/kaggle/working/ablation_results.csv

/kaggle/working/track_order.csv

In [9]:
# CELL 9 - PENGUJIAN FINAL SISTEM REKOMENDASI
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.preprocessing import normalize

# 1. Definisi Ulang Fungsinya (Untuk berjaga-jaga jika kernel restart)
def evaluate_ablation(alpha, beta, seed_song, seed_artist, n=10):
    # Gabungkan bobot
    weighted_audio = alpha * audio_matrix
    weighted_genre = beta * genre_matrix_norm
    combined = np.hstack([weighted_audio, weighted_genre]).astype(np.float32)
    combined = normalize(combined, norm='l2')
    
    # Cari index seed
    idx_list = df_metadata[
        (df_metadata['track_name'].str.contains(seed_song, case=False, na=False)) & 
        (df_metadata['artists'].str.contains(seed_artist, case=False, na=False))
    ].index
    
    if len(idx_list) == 0:
        print("Lagu tidak ditemukan!")
        return
        
    seed_idx = idx_list[0]
    seed_vector = combined[seed_idx].reshape(1, -1)
    
    # Hitung jarak
    sims = cosine_similarity(seed_vector, combined)[0]
    top_indices = sims.argsort()[::-1][1:n+1]
    
    print(f"\n[Ablation] α={alpha:.1f} (Audio) | β={beta:.1f} (Genre)")
    print(f"Seed: {df_metadata.iloc[seed_idx]['track_name']} - {df_metadata.iloc[seed_idx]['artists']}")
    print("-" * 70)
    for i, idx in enumerate(top_indices, 1):
        match = sims[idx] * 100
        print(f"{i}. [{match:.1f}%] {df_metadata.iloc[idx]['track_name'][:30]:<30} | {df_metadata.iloc[idx]['artists'][:20]:<20}")

# 2. Eksekusi Pencarian
seed_judul = "Baby Blue Eyes"
seed_musisi = "A Rocket to the Moon"

print("🔍 Memulai pencarian dengan Matriks Hybrid (50% Audio + 50% Genre)")
evaluate_ablation(0.5, 0.5, seed_judul, seed_musisi, n=10)

🔍 Memulai pencarian dengan Matriks Hybrid (50% Audio + 50% Genre)

[Ablation] α=0.5 (Audio) | β=0.5 (Genre)
Seed: Baby Blue Eyes - ["A Rocket To The Moon"]
----------------------------------------------------------------------
1. [98.1%] Into Your Arms                 | ["The Maine"]       
2. [98.0%] Slip the Noose                 | ["The Maine"]       
3. [97.9%] First Kiss                     | ["A Rocket To The Mo
4. [97.8%] Pictures                       | ["Count The Stars"] 
5. [97.7%] Destiny                        | ["The Rocket Summer"
6. [97.7%] You've Got It Made             | ["We Are The In Crow
7. [97.5%] deadfriends                    | ["Daytrader"]       
8. [97.5%] Russian House Dj               | ["Mixtapes"]        
9. [97.2%] Like We Used To                | ["A Rocket To The Mo
10. [96.9%] Bathwater                      | ["Tonight Alive"]   


In [10]:
# CELL 9 - PENGUJIAN FINAL SISTEM REKOMENDASI
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.preprocessing import normalize

# 1. Definisi Ulang Fungsinya (Untuk berjaga-jaga jika kernel restart)
def evaluate_ablation(alpha, beta, seed_song, seed_artist, n=10):
    # Gabungkan bobot
    weighted_audio = alpha * audio_matrix
    weighted_genre = beta * genre_matrix_norm
    combined = np.hstack([weighted_audio, weighted_genre]).astype(np.float32)
    combined = normalize(combined, norm='l2')
    
    # Cari index seed
    idx_list = df_metadata[
        (df_metadata['track_name'].str.contains(seed_song, case=False, na=False)) & 
        (df_metadata['artists'].str.contains(seed_artist, case=False, na=False))
    ].index
    
    if len(idx_list) == 0:
        print("Lagu tidak ditemukan!")
        return
        
    seed_idx = idx_list[0]
    seed_vector = combined[seed_idx].reshape(1, -1)
    
    # Hitung jarak
    sims = cosine_similarity(seed_vector, combined)[0]
    top_indices = sims.argsort()[::-1][1:n+1]
    
    print(f"\n[Ablation] α={alpha:.1f} (Audio) | β={beta:.1f} (Genre)")
    print(f"Seed: {df_metadata.iloc[seed_idx]['track_name']} - {df_metadata.iloc[seed_idx]['artists']}")
    print("-" * 70)
    for i, idx in enumerate(top_indices, 1):
        match = sims[idx] * 100
        print(f"{i}. [{match:.1f}%] {df_metadata.iloc[idx]['track_name'][:30]:<30} | {df_metadata.iloc[idx]['artists'][:20]:<20}")

# 2. Eksekusi Pencarian
seed_judul = "Shape of you"
seed_musisi = "Ed Sheeran"

print("🔍 Memulai pencarian dengan Matriks Hybrid (50% Audio + 50% Genre)")
evaluate_ablation(0.5, 0.5, seed_judul, seed_musisi, n=10)

🔍 Memulai pencarian dengan Matriks Hybrid (50% Audio + 50% Genre)

[Ablation] α=0.5 (Audio) | β=0.5 (Genre)
Seed: Shape of You - Acoustic - ["Ed Sheeran"]
----------------------------------------------------------------------
1. [93.2%] What Do I Know?                | ["Ed Sheeran"]      
2. [92.8%] Shape of You                   | ["Ed Sheeran"]      
3. [92.7%] Touch and Go                   | ["Ed Sheeran"]      
4. [92.0%] Light Switch - Acoustic        | ["Charlie Puth"]    
5. [91.4%] Marvin Gaye (feat. Meghan Trai | ["Charlie Puth", "Me
6. [91.1%] Big Racks (feat. Brooke Candy) | ["Bree Runway", "Bro
7. [91.0%] The Elf by the Shelf           | ["Kids Christmas Par
8. [90.9%] 12 Hours                       | ["Chris James"]     
9. [90.4%] All Work and No Play           | ["Van Morrison"]    
10. [90.1%] Empty Cups                     | ["Charlie Puth"]    


In [11]:
# CELL 9 - PENGUJIAN FINAL SISTEM REKOMENDASI (DENGAN FILTER ANTI-VARIAN)
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.preprocessing import normalize

def evaluate_ablation(alpha, beta, seed_song, seed_artist, n=10):
    weighted_audio = alpha * audio_matrix
    weighted_genre = beta * genre_matrix_norm
    combined = np.hstack([weighted_audio, weighted_genre]).astype(np.float32)
    combined = normalize(combined, norm='l2')
    
    idx_list = df_metadata[
        (df_metadata['track_name'].str.contains(seed_song, case=False, na=False)) & 
        (df_metadata['artists'].str.contains(seed_artist, case=False, na=False))
    ].index
    
    if len(idx_list) == 0:
        print("Lagu tidak ditemukan!")
        return
        
    seed_idx = idx_list[0]
    seed_vector = combined[seed_idx].reshape(1, -1)
    
    # Hitung jarak dengan seluruh lagu
    sims = cosine_similarity(seed_vector, combined)[0]
    
    # Urutkan dari yang paling mirip
    top_indices_all = sims.argsort()[::-1]
    
    print(f"\n[Ablation] α={alpha:.1f} (Audio) | β={beta:.1f} (Genre)")
    
    # Simpan nama lagu seed asli untuk difilter
    seed_track_name = df_metadata.iloc[seed_idx]['track_name'].lower()
    
    print(f"Seed: {df_metadata.iloc[seed_idx]['track_name']} - {df_metadata.iloc[seed_idx]['artists']}")
    print("-" * 70)
    
    # FILTERING PROCESS
    recommendations_printed = 0
    for idx in top_indices_all:
        if idx == seed_idx:
            continue # Skip lagu itu sendiri
            
        nama_lagu = df_metadata.iloc[idx]['track_name']
        artis = df_metadata.iloc[idx]['artists']
        
        # FILTER: Skip jika judulnya mengandung judul lagu seed (atau sebaliknya)
        # Ini akan membuang versi Acoustic, Live, Radio Edit, Remix, dll
        if seed_song.lower() in nama_lagu.lower() or nama_lagu.lower() in seed_song.lower():
            continue
            
        match = sims[idx] * 100
        recommendations_printed += 1
        print(f"{recommendations_printed}. [{match:.1f}%] {nama_lagu[:30]:<30} | {artis[:20]:<20}")
        
        if recommendations_printed == n:
            break

# Eksekusi Pencarian
seed_judul = "Shape of you"
seed_musisi = "Ed Sheeran"

print("🔍 Memulai pencarian dengan Matriks Hybrid (50% Audio + 50% Genre)")
evaluate_ablation(0.5, 0.5, seed_judul, seed_musisi, n=10)

🔍 Memulai pencarian dengan Matriks Hybrid (50% Audio + 50% Genre)

[Ablation] α=0.5 (Audio) | β=0.5 (Genre)
Seed: Shape of You - Acoustic - ["Ed Sheeran"]
----------------------------------------------------------------------
1. [93.2%] What Do I Know?                | ["Ed Sheeran"]      
2. [92.7%] Touch and Go                   | ["Ed Sheeran"]      
3. [92.0%] Light Switch - Acoustic        | ["Charlie Puth"]    
4. [91.4%] Marvin Gaye (feat. Meghan Trai | ["Charlie Puth", "Me
5. [91.1%] Big Racks (feat. Brooke Candy) | ["Bree Runway", "Bro
6. [91.0%] The Elf by the Shelf           | ["Kids Christmas Par
7. [90.9%] 12 Hours                       | ["Chris James"]     
8. [90.4%] All Work and No Play           | ["Van Morrison"]    
9. [90.1%] Empty Cups                     | ["Charlie Puth"]    
10. [90.0%] Tournament Hill                | ["TEMPOREX"]        
